# Testing the FOLDE Module Machinery

This notebook tests the functionality of the FOLDE (Few-shot and zerO-shot Learning for protein Design and Engineering) module, particularly focusing on the data loading functionality.

In [4]:
%load_ext autoreload
%autoreload 2
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import seaborn as sns

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

from folde.data import get_available_proteingym_datasets, get_proteingym_dataset
from folde.campaign import simulate_campaign, simulate_campaigns, simulate_campaigns_with_config_checkpoints
from folde.types import FolDEModelConfig, ModelEvaluation, ModelDiff
from folde.util import apply_diff_list_to_config

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:

EMBEDDING_MODEL_ID = '300m_extras'
NATURALNESS_MODEL_ID = '600m'

print(f"Testing with embedding model: {EMBEDDING_MODEL_ID} and naturalness model: {NATURALNESS_MODEL_ID}")
available_datasets = get_available_proteingym_datasets(EMBEDDING_MODEL_ID, NATURALNESS_MODEL_ID)

assert not available_datasets.empty
print(f"Found {len(available_datasets)} available datasets")
print("\nSample of available datasets:")
display(available_datasets.head(10))

2025-05-21 19:22:25,267 - folde.data - INFO - Loaded metadata for 217 DMS datasets
2025-05-21 19:22:25,325 - folde.data - INFO - Found 9 datasets with embedding model '300m_extras' and naturalness model '600m'


Testing with embedding model: 300m_extras and naturalness model: 600m
Found 9 available datasets

Sample of available datasets:


,DMS_id,DMS_filename,UniProt_ID,taxon,source_organism,target_seq,seq_len,includes_multiple_mutants,DMS_total_number_mutants,DMS_number_single_mutants,...,raw_DMS_filename,raw_DMS_phenotype_name,raw_DMS_directionality,raw_DMS_mutant_column,weight_file_name,pdb_file,pdb_range,ProteinGym_version,raw_mut_offset,coarse_selection_type
15,ANCSZ_Hobbs_2022,ANCSZ_Hobbs_2022.csv,ANCSZ,Eukaryote,Reconstructed ancestor,MADSANHLPYFYGSITREEAEDYLKQGGMSDGLFLLRQSLNSLGGY...,627,False,4670,4670,...,ANCSZ_Hobbs_2022.csv,DMS_value,1,mutant,ANCSZ_theta_0.2.npy,ANCSZ.pdb,1-627,1.0,NaN,Activity
21,BLAT_ECOLX_Firnberg_2014,BLAT_ECOLX_Firnberg_2014.csv,BLAT_ECOLX,Prokaryote,Escherichia coli,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,286,False,4783,4783,...,BLAT_ECOLX_Firnberg_2014.csv,linear,1,mutant,BLAT_ECOLX_theta_0.2.npy,BLAT_ECOLX.pdb,1-286,0.1,NaN,OrganismalFitness
36,CBS_HUMAN_Sun_2020,CBS_HUMAN_Sun_2020.csv,CBS_HUMAN,Human,Homo sapiens,MPSETPQAEVGPTGCPHRSGPHSAKGSLEKGSPEDKEAKEPLWIRP...,551,False,7217,7217,...,NaN,score,1,mutant,CBS_HUMAN_theta0.2_2023-10-12_b08.npy,CBS_HUMAN.pdb,1-551,1.0,NaN,OrganismalFitness
72,HEM3_HUMAN_Loggerenberg_2023,HEM3_HUMAN_Loggerenberg_2023.csv,HEM3_HUMAN,Human,Homo sapiens,MSGNGNAAATAEENSPKMRVIRVGTRKSQLARIQTDSVVATLKASY...,361,False,5689,5689,...,NaN,score,1,mutant,HEM3_HUMAN_theta0.2_2023-08-07_b02.npy,HEM3_HUMAN.pdb,1-361,1.0,NaN,Activity
76,HSP82_YEAST_Flynn_2019,HSP82_YEAST_Flynn_2019.csv,HSP82_YEAST,Eukaryote,Saccharomyces cerevisiae,MASETFEFQAEITQLMSLIINTVYSNKEIFLRELISNASDALDKIR...,709,False,13294,13294,...,HSP82_YEAST_Flynn_2019.csv,s (37°C),1,mutant,HSP82_YEAST_theta_0.2.npy,HSP82_YEAST.pdb,1-709,1.0,NaN,OrganismalFitness
78,HXK4_HUMAN_Gersing_2022_activity,HXK4_HUMAN_Gersing_2022_activity.csv,HXK4_HUMAN,Human,Homo sapiens,MLDDRARMEAAKKEKVEQILAEFQLQEEDLKKVMRRMQKEMDRGLR...,465,False,8570,8570,...,HXK4_HUMAN_Gersing_2022.csv,score,1,mutant,HXK4_HUMAN_theta_0.2.npy,HXK4_HUMAN.pdb,1-465,1.0,NaN,OrganismalFitness
114,OXDA_RHOTO_Vanella_2023_activity,OXDA_RHOTO_Vanella_2023_activity.csv,OXDA_RHOTO,Eukaryote,Rhodotorula gracilis,HSQKRVVVLGSGVIGLSSALILARKGYSVHILARDLPEDVSSQTFA...,364,False,6396,6396,...,Figure_2.xlsx,activity fitness,1,mutant,OXDA_RHOTO_theta0.2_2023-08-07_b02.npy,OXDA_RHOTO.pdb,1-364,1.0,NaN,Activity
133,PPM1D_HUMAN_Miller_2022,PPM1D_HUMAN_Miller_2022.csv,PPM1D_HUMAN,Human,Homo sapiens,MAGLYSLGVSVFSDQGGRKYMEDVTQIVVEPEPTAEEKPSPRRSLS...,605,False,7889,7889,...,PPM1D_HUMAN_Miller_2022_raw.xlsx,fitness,1,mutant,PPM1D_HUMAN_theta0.2_2023-10-12_b01.npy,PPM1D_HUMAN.pdb,1-605,1.0,NaN,OrganismalFitness
178,SHOC2_HUMAN_Kwon_2022,SHOC2_HUMAN_Kwon_2022.csv,SHOC2_HUMAN,Human,Homo sapiens,MSSSLGKEKDSKEKDPKVPSAKEREKEAKASGGFGKESKEKEPKTK...,582,False,10972,10972,...,2022.3.16.Extended Data Table 4.csv,LFC_scaled,1,variant.by.aa,SHOC2_HUMAN_theta0.2_2023-10-12_b04.npy,SHOC2_HUMAN.pdb,1-582,1.0,NaN,OrganismalFitness


# Configure Campaigns

In [6]:

# Example configuration
NAME = '250521_new_weights_who_this'

base_config = FolDEModelConfig(
        name="FolDE",
        # Required parameters
        naturalness_model_id="600m",  # ESM-2 650M model
        embedding_model_id="300m_extras",  # Same model for embeddings
        zero_shot_model_name="NaturalnessZeroShotModel",
        zero_shot_model_params={},
        # Few-shot model configuration (used after first round)
        few_shot_model_name="TorchMLPFewShotModel",
        few_shot_model_params={
            "pretrain": True,
            "pretrain_epochs": 50,

            "ensemble_size": 5,
            "decision_mode": "ucb",

            "embedding_dim": 960,
            "hidden_dims": [100, 50],
            "dropout": 0.2,
            "learning_rate": 0.001,
            "weight_decay": 1e-5,
            "train_epochs": 200,
            "train_patience": 1000,
            "val_frequency": 10,
        },
    )

config_list = apply_diff_list_to_config(
    base_config,
    [
        ModelDiff(
            name="holdout",
            diffs={
                "few_shot_model_params.do_holdout_validation": True,
            }
        ),
        ModelDiff(
            name="holdout_no_ucb",
            diffs={
                "few_shot_model_params.do_holdout_validation": True,
                "few_shot_model_params.decision_mode": "mean",
            }
        ),
        ModelDiff(
            name="reweight_min_T5",
            diffs={
                "few_shot_model_params.importance_sampling_reweighting_strat": 'min',
                "few_shot_model_params.importance_sampling_temperature": 5.0
            }
        ),
        ModelDiff(
            name="reweight_min_T10",
            diffs={
                "few_shot_model_params.importance_sampling_reweighting_strat": 'min',
                "few_shot_model_params.importance_sampling_temperature": 10.0
            }
        ),
        ModelDiff(
            name="reweight_min_T20",
            diffs={
                "few_shot_model_params.importance_sampling_reweighting_strat": 'min',
                "few_shot_model_params.importance_sampling_temperature": 20.0
            }
        ),
        ModelDiff(
            name="reweight_min_holdout",
            diffs={
                "few_shot_model_params.importance_sampling_reweighting_strat": 'min',
                "few_shot_model_params.importance_sampling_temperature": 10.0,
                "few_shot_model_params.do_holdout_validation": True,
            }
        ),
        ModelDiff(
            name="reweight_max",
            diffs={
                "few_shot_model_params.importance_sampling_reweighting_strat": 'max',
                "few_shot_model_params.importance_sampling_temperature": 10.0
            }
        ),
    ]
)

print(f"Config 1/{len(config_list)}:")
print(config_list[0].model_dump_json(indent=2))

Config 1/8:
{
  "name": "FolDE_holdout",
  "naturalness_model_id": "600m",
  "embedding_model_id": "300m_extras",
  "embedding_column": null,
  "zero_shot_model_name": "NaturalnessZeroShotModel",
  "zero_shot_model_params": {},
  "few_shot_model_name": "TorchMLPFewShotModel",
  "few_shot_naturalness_column": null,
  "few_shot_model_params": {
    "pretrain": true,
    "pretrain_epochs": 50,
    "ensemble_size": 5,
    "decision_mode": "ucb",
    "embedding_dim": 960,
    "hidden_dims": [
      100,
      50
    ],
    "dropout": 0.2,
    "learning_rate": 0.001,
    "weight_decay": 0.00001,
    "train_epochs": 200,
    "train_patience": 1000,
    "val_frequency": 10,
    "do_holdout_validation": true
  }
}


In [ ]:
VIRUS_IDS = [
    'A0A140D2T1_ZIKV_Sourisseau_2019',
    'A0A2Z5U3Z0_9INFA_Doud_2016'
]
results = simulate_campaigns_with_config_checkpoints(
  eval_prefix=NAME,
  dms_ids=[v for v in available_datasets['DMS_id'].values if v not in VIRUS_IDS],
  config_list=config_list,
  checkpoint_dir="notebooks/jacob/model_evals",
  round_size=16,
  number_of_simulations=10,
  activity_column="DMS_score",
  max_rounds=6,
  random_seed=42,
)

2025-05-21 19:22:27,211 - folde.campaign - INFO - [FolDE_holdout] Simulating DMS 'ANCSZ_Hobbs_2022'.
2025-05-21 19:22:27,223 - folde.campaign - INFO - Running simulations for configuration 1/1
2025-05-21 19:22:27,224 - folde.campaign - INFO - Config: name='FolDE_holdout' naturalness_model_id='600m' embedding_model_id='300m_extras' embedding_column=None zero_shot_model_name='NaturalnessZeroShotModel' zero_shot_model_params={} few_shot_model_name='TorchMLPFewShotModel' few_shot_naturalness_column=None few_shot_model_params={'pretrain': True, 'pretrain_epochs': 50, 'ensemble_size': 5, 'decision_mode': 'ucb', 'embedding_dim': 960, 'hidden_dims': [100, 50], 'dropout': 0.2, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'train_epochs': 200, 'train_patience': 1000, 'val_frequency': 10, 'do_holdout_validation': True}


2025-05-21 19:22:27,345 - folde.data - INFO - Loaded activity data for ANCSZ_Hobbs_2022 with 4670 rows
2025-05-21 19:22:40,078 - folde.data - INFO - Loaded embeddings for ANCSZ_Hobbs_2022 with 11914 rows
2025-05-21 19:22:40,198 - folde.data - INFO - Loaded naturalness scores for ANCSZ_Hobbs_2022 with 20691 rows
2025-05-21 19:23:15,730 - folde.campaign - INFO - Running simulation 1 (2335 / 4670 mutants in sim)
2025-05-21 19:23:15,785 - root - INFO - Pretraining model with naturalness data with 2335 single mutants.
2025-05-21 19:23:16,581 - folde.campaign - INFO - Running simulation 2 (2335 / 4670 mutants in sim)
2025-05-21 19:23:16,653 - root - INFO - Pretraining model with naturalness data with 2335 single mutants.
2025-05-21 19:23:17,662 - folde.campaign - INFO - Running simulation 3 (2335 / 4670 mutants in sim)
2025-05-21 19:23:17,751 - root - INFO - Pretraining model with naturalness data with 2335 single mutants.
2025-05-21 19:23:18,501 - folde.campaign - INFO - Running simulation 

2025-05-21 19:25:01,955 - root - INFO - Finetune improvement: train loss (2.3949 -> 0.0403) val loss (0.4875 -> 1.7324)
2025-05-21 19:25:02,012 - root - INFO - Pretraining model with naturalness data with 2335 single mutants.
2025-05-21 19:25:02,156 - root - INFO - Finetune improvement: train loss (3.7985 -> 0.0599) val loss (0.2848 -> 0.7840)
2025-05-21 19:25:02,586 - root - INFO - Pretraining model with naturalness data with 2335 single mutants.
2025-05-21 19:25:03,842 - root - INFO - Finetune improvement: train loss (1.1863 -> 0.0325) val loss (0.3991 -> 0.5090)
2025-05-21 19:25:03,860 - root - INFO - Finetune improvement: train loss (2.8727 -> 0.0522) val loss (0.2980 -> 0.2440)
2025-05-21 19:25:03,917 - root - INFO - Finetune improvement: train loss (3.2756 -> 0.0337) val loss (0.2152 -> 0.3576)
2025-05-21 19:25:04,065 - root - INFO - Finetune improvement: train loss (3.5659 -> 0.0423) val loss (0.4160 -> 0.4296)
2025-05-21 19:25:04,130 - root - INFO - Finetune improvement: train 